In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Row
from pyspark.sql.types import *
from datetime import datetime
import uuid
import time

In [0]:
%sql
CREATE TABLE ETL_Audit_Log (
    job_run_id STRING,
    job_name STRING,
    job_start_time TIMESTAMP,
    job_end_time TIMESTAMP,
    execution_time_seconds DOUBLE,
    rows_processed INT,
    rows_rejected INT,
    status STRING
)
USING DELTA;

In [0]:
%sql
ALTER TABLE ETL_Audit_Log ADD COLUMNS (error_message STRING);

In [0]:
import uuid
import time

job_run_id = str(uuid.uuid4())
job_name = "financial_risk_pipeline"

start_time = time.time()

In [0]:
rows_processed = spark.read.table("gold_stock_risk").count()
rows_rejected = spark.read.table("silver_invalid_records").count()

status = "SUCCESS"
error_message = None

In [0]:
end_time = time.time()

execution_time = end_time - start_time

In [0]:
status = "SUCCESS"

In [0]:
from pyspark.sql.types import *

audit_schema = StructType([
    StructField("job_run_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_start_time", TimestampType(), True),
    StructField("job_end_time", TimestampType(), True),
    StructField("execution_time_seconds", DoubleType(), True),
    StructField("rows_processed", IntegerType(), True),
    StructField("rows_rejected", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)   # 👈 ADD THIS
])

In [0]:
from pyspark.sql import Row

audit_data = [Row(
    job_run_id=job_run_id,
    job_name=job_name,
    job_start_time=datetime.fromtimestamp(start_time),
    job_end_time=datetime.fromtimestamp(end_time),
    execution_time_seconds=float(execution_time),
    rows_processed=int(rows_processed),
    rows_rejected=int(rows_rejected),
    status=status,
    error_message=error_message
)]

In [0]:
audit_df = spark.createDataFrame(audit_data, schema=audit_schema)

In [0]:
audit_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("ETL_Audit_Log")

In [0]:
%sql
SELECT * FROM ETL_Audit_Log;

job_run_id,job_name,job_start_time,job_end_time,execution_time_seconds,rows_processed,rows_rejected,status,error_message
411522f1-c9d0-445d-a350-e531e53720c4,financial_risk_pipeline,2026-03-17T12:15:49.142Z,2026-03-17T12:18:23.635Z,154.49311470985413,30,0,SUCCESS,null
